# Mastering Imperfect Information with Deep Recurrent Q-Networks
## Ultimate Comparison: 4 DRQN vs 4 DQN vs Baselines

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/DLPW/blob/master/notebooks/DLPW_colab.ipynb)

**This notebook trains and compares 10 agents:**

**DRQN Variants (with LSTM memory):**
1. DRQN-Standard (two-phase)
2. DRQN-MultiTask (prevents forgetting)
3. DRQN-SelfPlay (Nash equilibrium)
4. DRQN-Hybrid (best of both)

**DQN Variants (memoryless baseline):**
5. DQN-Standard (two-phase)
6. DQN-MultiTask
7. DQN-SelfPlay
8. DQN-Hybrid

**Baselines:**
9. Random Agent
10. Heuristic Agent (rule-based)

**Evaluations:**
- All 8 models vs Random (8 matchups)
- All 8 models vs Heuristic (8 matchups)
- All DRQN vs all DQN cross-evaluation (4×4 = 16 matchups)
- DRQN variants vs each other (6 matchups)
- DQN variants vs each other (6 matchups)
- **Total: 44 matchups**

**Runtime**: GPU recommended (~4-5 hours for all 8 models)

## 1. Setup & Installation

In [ ]:
!pip install -q rlcard torch matplotlib pandas

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Dependencies installed")
print(f"✓ Device: {device}")
print(f"✓ PyTorch version: {torch.__version__}")

## 2. Clone Repository

In [ ]:
import os
import sys

# Clone repository (change YOUR_USERNAME)
!git clone https://github.com/YOUR_USERNAME/DLPW.git
%cd DLPW

sys.path.insert(0, '/content/DLPW')
print("✓ Repository cloned and path configured")

## 3. Configuration

In [ ]:
import config

# Adjust for Colab (increase for better results)
config.NUM_EPISODES_PHASE1 = 8000
config.NUM_EPISODES_PHASE2 = 8000
config.EVALUATE_EVERY = 1000
config.EVALUATE_NUM = 500
config.NUM_EVAL_HANDS = 1000

# Paths
config.OUTPUT_DIR = '/content/DLPW/outputs'
config.MODEL_DIR = '/content/DLPW/outputs/models'
config.PLOT_DIR = '/content/DLPW/outputs/plots'

os.makedirs(config.OUTPUT_DIR, exist_ok=True)
os.makedirs(config.MODEL_DIR, exist_ok=True)
os.makedirs(config.PLOT_DIR, exist_ok=True)

print("✓ Configuration set")
print(f"  Episodes per phase: {config.NUM_EPISODES_PHASE1}")
print(f"  Expected time: ~4-5 hours on GPU (8 models)")

## 4. Import Modules

In [ ]:
import rlcard
from rlcard.utils import set_seed
from rlcard.agents import RandomAgent, DQNAgent
import copy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import logging

from config import *
from agents import DRQNAgent, ConservativeHeuristicAgent
from training import CurriculumTrainer
from evaluation import evaluate_agents, compute_action_distribution

# Suppress RLCard INFO logs
logging.getLogger('rlcard').setLevel(logging.WARNING)

print("✓ All modules imported")

## 5. Environment Setup

In [ ]:
set_seed(SEED)
env = rlcard.make(ENV_NAME)

raw_shape = env.state_shape[0]
state_shape = raw_shape[0] if isinstance(raw_shape, list) else raw_shape
num_actions = env.num_actions

# Baseline agents
agent_random = RandomAgent(num_actions)
agent_heuristic = ConservativeHeuristicAgent(num_actions)

print(f'✓ Environment: {ENV_NAME}')
print(f'✓ State shape: {state_shape}, Actions: {num_actions}')

## 6. Training Function (for both DRQN and DQN)

In [ ]:
def train_agent_variant(agent, agent_type, variant_name, 
                       use_self_play=False, multi_task=False):
    """
    Train an agent with specific configuration.
    
    Args:
        agent: DRQN or DQN agent
        agent_type: 'DRQN' or 'DQN'
        variant_name: 'Standard', 'MultiTask', 'SelfPlay', 'Hybrid'
        use_self_play: Whether to use self-play
        multi_task: Whether to use multi-task learning
    
    Returns:
        history: Training history
    """
    print(f"\n{'='*70}")
    print(f"TRAINING {agent_type}-{variant_name.upper()}")
    print(f"{'='*70}")
    
    if variant_name == 'Hybrid':
        # Phase 1: Multi-task
        trainer_p1 = CurriculumTrainer(
            env=env, agent=agent,
            opponent_phase1=agent_random,
            opponent_phase2=agent_heuristic,
            use_self_play=False,
            multi_task=True,
            multi_task_ratio=0.5
        )
        trainer_p1.train_phase1(NUM_EPISODES_PHASE1, EVALUATE_EVERY, EVALUATE_NUM)
        
        # Phase 2: Self-play
        trainer_p2 = CurriculumTrainer(
            env=env, agent=agent,
            opponent_phase1=agent_random,
            opponent_phase2=agent_heuristic,
            use_self_play=True,
            multi_task=False
        )
        trainer_p2.ev_history_random = trainer_p1.ev_history_random
        trainer_p2.ev_history_heuristic = trainer_p1.ev_history_heuristic
        trainer_p2.loss_history = trainer_p1.loss_history
        trainer_p2.train_phase2(NUM_EPISODES_PHASE1, NUM_EPISODES_PHASE2,
                                EVALUATE_EVERY, EVALUATE_NUM, new_lr=LEARNING_RATE_PHASE2)
        
        history = {
            'ev_random': trainer_p2.ev_history_random,
            'ev_heuristic': trainer_p2.ev_history_heuristic,
            'loss': trainer_p2.loss_history
        }
    else:
        # Standard, MultiTask, or SelfPlay
        trainer = CurriculumTrainer(
            env=env, agent=agent,
            opponent_phase1=agent_random,
            opponent_phase2=agent_heuristic,
            use_self_play=use_self_play,
            multi_task=multi_task,
            multi_task_ratio=0.5 if multi_task else 0.0
        )
        history = trainer.train()
    
    # Save model
    model_name = f"{agent_type.lower()}_{variant_name.lower()}.pt"
    model_path = f"{MODEL_DIR}/{model_name}"
    
    if hasattr(agent, 'save_model'):
        agent.save_model(model_path)
        print(f"\n✓ Model saved: {model_path}")
    
    return history

print("✓ Training function defined")

## 7. Train All DRQN Variants (4 models)

In [ ]:
print("\n" + "="*70)
print("PHASE 1: TRAINING ALL DRQN VARIANTS")
print("="*70)

drqn_agents = {}
drqn_histories = {}

variants = [
    ('Standard', False, False),
    ('MultiTask', False, True),
    ('SelfPlay', True, False),
    ('Hybrid', None, None)  # Special handling in train_agent_variant
]

for variant_name, use_self_play, multi_task in variants:
    # Create agent
    agent = DRQNAgent(
        state_shape=state_shape,
        num_actions=num_actions,
        device=DEVICE,
        hidden_size=HIDDEN_SIZE,
        lr=LEARNING_RATE,
        gamma=GAMMA,
        epsilon_start=EPSILON_START,
        epsilon_min=EPSILON_MIN,
        epsilon_decay=EPSILON_DECAY,
        buffer_capacity=BUFFER_CAPACITY,
        batch_size=BATCH_SIZE,
        min_replay=MIN_REPLAY_SIZE,
        target_update_freq=TARGET_UPDATE_FREQ,
        l2_reg=L2_REGULARIZATION
    )
    
    # Train
    history = train_agent_variant(
        agent, 'DRQN', variant_name,
        use_self_play=use_self_play if use_self_play is not None else False,
        multi_task=multi_task if multi_task is not None else False
    )
    
    drqn_agents[variant_name] = agent
    drqn_histories[variant_name] = history

print("\n✓ All DRQN variants trained")

## 8. Train All DQN Variants (4 models)

In [ ]:
print("\n" + "="*70)
print("PHASE 2: TRAINING ALL DQN VARIANTS")
print("="*70)

dqn_agents = {}
dqn_histories = {}

for variant_name, use_self_play, multi_task in variants:
    # Create DQN agent (from RLCard)
    agent = DQNAgent(
        num_actions=num_actions,
        state_shape=env.state_shape[0],
        mlp_layers=[HIDDEN_SIZE, HIDDEN_SIZE],
        device=DEVICE
    )
    
    # Train
    history = train_agent_variant(
        agent, 'DQN', variant_name,
        use_self_play=use_self_play if use_self_play is not None else False,
        multi_task=multi_task if multi_task is not None else False
    )
    
    dqn_agents[variant_name] = agent
    dqn_histories[variant_name] = history

print("\n✓ All DQN variants trained")
print("\n" + "="*70)
print("ALL TRAINING COMPLETE (8 models)")
print("="*70)

## 9. Comprehensive Evaluation

In [ ]:
print("\n" + "="*70)
print("COMPREHENSIVE EVALUATION - ALL 10 AGENTS")
print(f"({NUM_EVAL_HANDS} hands per matchup)")
print("="*70)

# Collect all agents
all_agents = {
    'Random': agent_random,
    'Heuristic': agent_heuristic
}

for variant in ['Standard', 'MultiTask', 'SelfPlay', 'Hybrid']:
    all_agents[f'DRQN-{variant}'] = drqn_agents[variant]
    all_agents[f'DQN-{variant}'] = dqn_agents[variant]

# Store results
evaluation_results = {}

# Evaluate vs baselines
print("\n1. EVALUATION VS BASELINES")
print("="*70)

for agent_name in all_agents.keys():
    if agent_name in ['Random', 'Heuristic']:
        continue
    
    print(f"\n{agent_name}:")
    evaluation_results[agent_name] = {}
    
    for baseline_name in ['Random', 'Heuristic']:
        ev, traj = evaluate_agents(
            env, all_agents[agent_name], all_agents[baseline_name],
            NUM_EVAL_HANDS, f'  vs {baseline_name}'
        )
        
        stats = compute_action_distribution(traj, ACTION_NAMES)
        evaluation_results[agent_name][baseline_name] = {
            'ev': ev,
            'stats': stats
        }

print("\n✓ Baseline evaluation complete")

## 10. All DRQN vs All DQN Cross-Evaluation (16 matchups)

In [ ]:
print("\n" + "="*70)
print("2. ALL DRQN vs ALL DQN CROSS-EVALUATION (16 MATCHUPS)")
print("="*70)
print("\nEvaluating every DRQN variant against every DQN variant...")

# 4 DRQN x 4 DQN = 16 matchups
for drqn_variant in ['Standard', 'MultiTask', 'SelfPlay', 'Hybrid']:
    for dqn_variant in ['Standard', 'MultiTask', 'SelfPlay', 'Hybrid']:
        drqn_name = f'DRQN-{drqn_variant}'
        dqn_name = f'DQN-{dqn_variant}'
        
        ev, _ = evaluate_agents(
            env, drqn_agents[drqn_variant], dqn_agents[dqn_variant],
            NUM_EVAL_HANDS, f'{drqn_name} vs {dqn_name}'
        )
        
        # Store result
        if drqn_name not in evaluation_results:
            evaluation_results[drqn_name] = {}
        evaluation_results[drqn_name][dqn_name] = {'ev': ev}

print("\n✓ Cross-evaluation complete (16 matchups)")

# Create cross-evaluation matrix for visualization
print("\n" + "="*70)
print("DRQN vs DQN CROSS-EVALUATION MATRIX")
print("(Rows: DRQN variants, Columns: DQN variants, Values: EV from DRQN perspective)")
print("="*70)

matrix_data = []
for drqn_v in ['Standard', 'MultiTask', 'SelfPlay', 'Hybrid']:
    row = {'DRQN': drqn_v}
    for dqn_v in ['Standard', 'MultiTask', 'SelfPlay', 'Hybrid']:
        drqn_name = f'DRQN-{drqn_v}'
        dqn_name = f'DQN-{dqn_v}'
        ev = evaluation_results[drqn_name][dqn_name]['ev']
        row[f'vs DQN-{dqn_v}'] = f"{ev:+.3f}"
    matrix_data.append(row)

df_matrix = pd.DataFrame(matrix_data)
print("\n" + df_matrix.to_string(index=False))

# Summary statistics
print("\n" + "="*70)
print("CROSS-EVALUATION SUMMARY")
print("="*70)

drqn_wins = 0
dqn_wins = 0
ties = 0

for drqn_v in ['Standard', 'MultiTask', 'SelfPlay', 'Hybrid']:
    for dqn_v in ['Standard', 'MultiTask', 'SelfPlay', 'Hybrid']:
        drqn_name = f'DRQN-{drqn_v}'
        dqn_name = f'DQN-{dqn_v}'
        ev = evaluation_results[drqn_name][dqn_name]['ev']
        
        if ev > 0.05:
            drqn_wins += 1
        elif ev < -0.05:
            dqn_wins += 1
        else:
            ties += 1

print(f"\nOut of 16 matchups:")
print(f"  DRQN wins: {drqn_wins} ({drqn_wins/16*100:.1f}%)")
print(f"  DQN wins:  {dqn_wins} ({dqn_wins/16*100:.1f}%)")
print(f"  Ties:      {ties} ({ties/16*100:.1f}%)")

if drqn_wins > dqn_wins:
    print(f"\n✓ Overall: DRQN dominates with {drqn_wins}/{16} wins")
elif dqn_wins > drqn_wins:
    print(f"\n✗ Overall: DQN dominates with {dqn_wins}/{16} wins")
else:
    print(f"\n≈ Overall: Balanced matchup")

## 11. DRQN vs DRQN Comparison (6 matchups)

In [ ]:
print("\n" + "="*70)
print("3. DRQN VARIANTS COMPARISON")
print("="*70)

drqn_variants = ['Standard', 'MultiTask', 'SelfPlay', 'Hybrid']
for i, v1 in enumerate(drqn_variants):
    for v2 in drqn_variants[i+1:]:
        name1 = f'DRQN-{v1}'
        name2 = f'DRQN-{v2}'

        ev, _ = evaluate_agents(
            env, drqn_agents[v1], drqn_agents[v2],
            NUM_EVAL_HANDS, f'{name1} vs {name2}'
        )

        if name1 not in evaluation_results:
            evaluation_results[name1] = {}
        evaluation_results[name1][name2] = {'ev': ev}

print("\n✓ DRQN comparison complete")

## 12. DQN vs DQN Comparison (6 matchups)

In [ ]:
print("\n" + "="*70)
print("4. DQN VARIANTS COMPARISON")
print("="*70)

dqn_variants = ['Standard', 'MultiTask', 'SelfPlay', 'Hybrid']
for i, v1 in enumerate(dqn_variants):
    for v2 in dqn_variants[i+1:]:
        name1 = f'DQN-{v1}'
        name2 = f'DQN-{v2}'
        
        ev, _ = evaluate_agents(
            env, dqn_agents[v1], dqn_agents[v2],
            NUM_EVAL_HANDS, f'{name1} vs {name2}'
        )
        
        if name1 not in evaluation_results:
            evaluation_results[name1] = {}
        evaluation_results[name1][name2] = {'ev': ev}

print("\n✓ DQN comparison complete")

## 13. Results Summary Table

In [ ]:
print("\n" + "="*70)
print("RESULTS SUMMARY TABLE")
print("="*70)

# Create comprehensive table
table_data = []

for variant in ['Standard', 'MultiTask', 'SelfPlay', 'Hybrid']:
    drqn_name = f'DRQN-{variant}'
    dqn_name = f'DQN-{variant}'
    
    # DRQN row
    drqn_row = {
        'Model': drqn_name,
        'vs Random': f"{evaluation_results[drqn_name]['Random']['ev']:+.3f}",
        'vs Heuristic': f"{evaluation_results[drqn_name]['Heuristic']['ev']:+.3f}",
        'vs DQN-Same': f"{evaluation_results[drqn_name][dqn_name]['ev']:+.3f}",
        'Bluff Rate': f"{evaluation_results[drqn_name]['Random']['stats']['bluff_rate']:.1f}%"
    }
    table_data.append(drqn_row)
    
    # DQN row
    dqn_row = {
        'Model': dqn_name,
        'vs Random': f"{evaluation_results[dqn_name]['Random']['ev']:+.3f}",
        'vs Heuristic': f"{evaluation_results[dqn_name]['Heuristic']['ev']:+.3f}",
        'vs DQN-Same': '-',
        'Bluff Rate': f"{evaluation_results[dqn_name]['Random']['stats']['bluff_rate']:.1f}%"
    }
    table_data.append(dqn_row)

df = pd.DataFrame(table_data)
print("\n" + df.to_string(index=False))
print("\n")

## 14. Memory Advantage Analysis

In [ ]:
print("\n" + "="*70)
print("MEMORY ADVANTAGE ANALYSIS (DRQN vs DQN)")
print("="*70)

for variant in ['Standard', 'MultiTask', 'SelfPlay', 'Hybrid']:
    drqn_name = f'DRQN-{variant}'
    dqn_name = f'DQN-{variant}'
    
    head_to_head = evaluation_results[drqn_name][dqn_name]['ev']
    
    # Overall score (weighted)
    drqn_score = (
        0.4 * evaluation_results[drqn_name]['Random']['ev'] +
        0.6 * evaluation_results[drqn_name]['Heuristic']['ev']
    )
    dqn_score = (
        0.4 * evaluation_results[dqn_name]['Random']['ev'] +
        0.6 * evaluation_results[dqn_name]['Heuristic']['ev']
    )
    
    advantage = drqn_score - dqn_score
    status = "✓ DRQN wins" if advantage > 0.1 else "≈ Tie" if abs(advantage) <= 0.1 else "✗ DQN wins"
    
    print(f"\n{variant}:")
    print(f"  Head-to-head:    {head_to_head:+.3f} (DRQN perspective)")
    print(f"  DRQN score:      {drqn_score:+.3f}")
    print(f"  DQN score:       {dqn_score:+.3f}")
    print(f"  Memory advantage: {advantage:+.3f} {status}")

## 15. Visualization: Training Curves

In [ ]:
# Plot DRQN vs DQN training curves
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Training Curves: DRQN vs DQN (All Variants)', fontsize=16, fontweight='bold')

for idx, variant in enumerate(['Standard', 'MultiTask', 'SelfPlay', 'Hybrid']):
    # DRQN plot
    ax_drqn = axes[0, idx]
    hist = drqn_histories[variant]
    
    if hist['ev_random']:
        eps, evs = zip(*hist['ev_random'])
        ax_drqn.plot(eps, evs, 'b-', label='vs Random', linewidth=2)
    if hist['ev_heuristic']:
        eps, evs = zip(*hist['ev_heuristic'])
        ax_drqn.plot(eps, evs, 'g--', label='vs Heuristic', linewidth=2)
    
    ax_drqn.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    ax_drqn.axvline(NUM_EPISODES_PHASE1, color='orange', linestyle=':', linewidth=1.5)
    ax_drqn.set_title(f'DRQN-{variant}', fontweight='bold')
    ax_drqn.set_xlabel('Episode')
    ax_drqn.set_ylabel('EV')
    ax_drqn.legend()
    ax_drqn.grid(True, alpha=0.3)
    
    # DQN plot
    ax_dqn = axes[1, idx]
    hist = dqn_histories[variant]
    
    if hist['ev_random']:
        eps, evs = zip(*hist['ev_random'])
        ax_dqn.plot(eps, evs, 'r-', label='vs Random', linewidth=2)
    if hist['ev_heuristic']:
        eps, evs = zip(*hist['ev_heuristic'])
        ax_dqn.plot(eps, evs, 'm--', label='vs Heuristic', linewidth=2)
    
    ax_dqn.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    ax_dqn.axvline(NUM_EPISODES_PHASE1, color='orange', linestyle=':', linewidth=1.5)
    ax_dqn.set_title(f'DQN-{variant}', fontweight='bold')
    ax_dqn.set_xlabel('Episode')
    ax_dqn.set_ylabel('EV')
    ax_dqn.legend()
    ax_dqn.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/all_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Training curves saved")

## 16. Visualization: Final Performance Comparison

In [ ]:
# Create comparison bar chart
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Final Performance: DRQN vs DQN (All Variants)', fontsize=14, fontweight='bold')

variants_list = ['Standard', 'MultiTask', 'SelfPlay', 'Hybrid']
x = np.arange(len(variants_list))
width = 0.35

# Plot 1: vs Random
drqn_random = [evaluation_results[f'DRQN-{v}']['Random']['ev'] for v in variants_list]
dqn_random = [evaluation_results[f'DQN-{v}']['Random']['ev'] for v in variants_list]

axes[0].bar(x - width/2, drqn_random, width, label='DRQN', color='blue', alpha=0.7)
axes[0].bar(x + width/2, dqn_random, width, label='DQN', color='red', alpha=0.7)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_ylabel('EV (chips/hand)')
axes[0].set_title('vs Random Agent')
axes[0].set_xticks(x)
axes[0].set_xticklabels(variants_list)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Plot 2: vs Heuristic
drqn_heuristic = [evaluation_results[f'DRQN-{v}']['Heuristic']['ev'] for v in variants_list]
dqn_heuristic = [evaluation_results[f'DQN-{v}']['Heuristic']['ev'] for v in variants_list]

axes[1].bar(x - width/2, drqn_heuristic, width, label='DRQN', color='blue', alpha=0.7)
axes[1].bar(x + width/2, dqn_heuristic, width, label='DQN', color='red', alpha=0.7)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_ylabel('EV (chips/hand)')
axes[1].set_title('vs Heuristic Agent')
axes[1].set_xticks(x)
axes[1].set_xticklabels(variants_list)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/final_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Comparison plot saved")

## 17. Winner Determination

In [ ]:
print("\n" + "="*70)
print("FINAL RANKINGS")
print("="*70)

# Calculate scores for all models
scores = {}

for variant in ['Standard', 'MultiTask', 'SelfPlay', 'Hybrid']:
    for model_type in ['DRQN', 'DQN']:
        model_name = f'{model_type}-{variant}'
        
        ev_random = evaluation_results[model_name]['Random']['ev']
        ev_heuristic = evaluation_results[model_name]['Heuristic']['ev']
        
        # Weighted score
        score = 0.4 * ev_random + 0.6 * ev_heuristic
        scores[model_name] = score

# Overall rankings
ranked_all = sorted(scores.items(), key=lambda x: -x[1])

print("\n1. OVERALL RANKINGS:")
for rank, (model, score) in enumerate(ranked_all, 1):
    medal = "🥇" if rank == 1 else "🥈" if rank == 2 else "🥉" if rank == 3 else "  "
    print(f"{medal} {rank}. {model:18s} Score: {score:+.3f}")

# Best DRQN vs Best DQN
drqn_scores = {k: v for k, v in scores.items() if k.startswith('DRQN')}
dqn_scores = {k: v for k, v in scores.items() if k.startswith('DQN')}

best_drqn = max(drqn_scores.items(), key=lambda x: x[1])
best_dqn = max(dqn_scores.items(), key=lambda x: x[1])

print(f"\n2. BEST DRQN: {best_drqn[0]} (Score: {best_drqn[1]:+.3f})")
print(f"3. BEST DQN:  {best_dqn[0]} (Score: {best_dqn[1]:+.3f})")

advantage = best_drqn[1] - best_dqn[1]
print(f"\n4. MEMORY ADVANTAGE: {advantage:+.3f}")

if advantage > 0.1:
    print("   ✓ DRQN significantly outperforms DQN")
    print("   ✓ Memory (LSTM) provides clear advantage")
elif advantage > 0:
    print("   ≈ DRQN slightly outperforms DQN")
    print("   ≈ Memory provides marginal advantage")
else:
    print("   ✗ DQN matches or exceeds DRQN")
    print("   ✗ Memory not beneficial (need more training)")

print(f"\n{'='*70}")
print(f"🏆 ULTIMATE WINNER: {ranked_all[0][0]}")
print(f"{'='*70}")

## 18. Key Insights

In [ ]:
print("\n" + "="*70)
print("KEY INSIGHTS")
print("="*70)

print("\n1. DOES MULTI-TASK PREVENT CATASTROPHIC FORGETTING?")
for model_type in ['DRQN', 'DQN']:
    std_name = f'{model_type}-Standard'
    mt_name = f'{model_type}-MultiTask'
    
    std_random = evaluation_results[std_name]['Random']['ev']
    mt_random = evaluation_results[mt_name]['Random']['ev']
    
    improvement = mt_random - std_random
    status = "✓" if improvement > 0.05 else "≈" if improvement > -0.05 else "✗"
    print(f"  {model_type}: {improvement:+.3f} {status}")

print("\n2. DOES SELF-PLAY WORK BETTER THAN HEURISTIC?")
for model_type in ['DRQN', 'DQN']:
    std_name = f'{model_type}-Standard'
    sp_name = f'{model_type}-SelfPlay'
    
    std_score = scores[std_name]
    sp_score = scores[sp_name]
    
    improvement = sp_score - std_score
    status = "✓" if improvement > 0.05 else "≈" if improvement > -0.05 else "✗"
    print(f"  {model_type}: {improvement:+.3f} {status}")

print("\n3. IS HYBRID (MULTI-TASK + SELF-PLAY) BEST?")
for model_type in ['DRQN', 'DQN']:
    hybrid_name = f'{model_type}-Hybrid'
    
    # Compare to others
    hybrid_score = scores[hybrid_name]
    type_scores = {k: v for k, v in scores.items() if k.startswith(model_type)}
    max_score = max(type_scores.values())
    
    is_best = hybrid_score == max_score
    status = "✓ YES" if is_best else f"✗ NO (best: {max(type_scores, key=type_scores.get).split('-')[1]})"
    print(f"  {model_type}: {status}")

print("\n4. DOES LSTM HELP ACROSS ALL TRAINING METHODS?")
consistent_wins = 0
for variant in ['Standard', 'MultiTask', 'SelfPlay', 'Hybrid']:
    drqn_name = f'DRQN-{variant}'
    dqn_name = f'DQN-{variant}'
    
    if scores[drqn_name] > scores[dqn_name]:
        consistent_wins += 1
        print(f"  {variant:12s}: ✓ DRQN wins")
    else:
        print(f"  {variant:12s}: ✗ DQN wins")

print(f"\n  Consistency: {consistent_wins}/4 variants favor DRQN")
if consistent_wins >= 3:
    print("  ✓ Memory provides robust advantage")
elif consistent_wins >= 2:
    print("  ≈ Memory helps in some cases")
else:
    print("  ✗ Memory not consistently beneficial")

## 19. Download All Results

## Summary

### Training Complete
✅ 4 DRQN variants trained  
✅ 4 DQN variants trained  
✅ 2 baseline agents  
**Total: 10 agents**

### Evaluation Complete
✅ All agents vs Random (8 matchups)  
✅ All agents vs Heuristic (8 matchups)  
✅ **All DRQN vs all DQN (16 matchups)**  
✅ DRQN variants vs each other (6 matchups)  
✅ DQN variants vs each other (6 matchups)  
**Total: 44 matchups**

### Key Findings
Check the rankings and insights above!

---

**Training Time**: ~4-5 hours on Colab GPU  
**Episodes**: 8K per phase (increase to 15K+ for production)

## Summary

### Training Complete
✅ 4 DRQN variants trained  
✅ 4 DQN variants trained  
✅ 2 baseline agents  
**Total: 10 agents**

### Evaluation Complete
✅ All agents vs Random  
✅ All agents vs Heuristic  
✅ DRQN vs DQN (same training)  
✅ DRQN variants vs each other  
✅ DQN variants vs each other  
**Total: 45+ matchups**

### Key Findings
Check the rankings and insights above!

---

**Training Time**: ~4-5 hours on Colab GPU  
**Episodes**: 8K per phase (increase to 15K+ for production)